In [1]:
import torch
import torchvision
import sys
import subprocess
import platform

In [2]:
def get_cpu_name():
    """Return the CPU model string, cross-platform (Windows / Linux / WSL / macOS)."""
    system = platform.system()
    try:
        if system == "Windows":
            output = subprocess.check_output("wmic cpu get name", shell=True).decode().strip()
            return output.split('\n')[-1].strip()
        elif system == "Linux":
            # Works on Linux and WSL: read the model name from /proc/cpuinfo
            with open("/proc/cpuinfo") as f:
                for line in f:
                    if line.lower().startswith("model name"):
                        return line.split(":", 1)[1].strip()
        elif system == "Darwin":  # macOS
            return subprocess.check_output(
                ["sysctl", "-n", "machdep.cpu.brand_string"]).decode().strip()
    except Exception:
        pass
    return platform.processor() or "Unknown"


def get_ram_gb():
    """Return total RAM in GB, cross-platform."""
    system = platform.system()
    try:
        if system == "Windows":
            output = subprocess.check_output(
                "wmic computersystem get TotalPhysicalMemory", shell=True).decode().strip()
            total_bytes = float(output.split('\n')[-1].strip())
            return round(total_bytes / (1024**3), 2)
        elif system == "Linux":
            # Works on Linux and WSL: MemTotal in /proc/meminfo is in kB
            with open("/proc/meminfo") as f:
                for line in f:
                    if line.startswith("MemTotal"):
                        total_kb = float(line.split()[1])
                        return round(total_kb / (1024**2), 2)
        elif system == "Darwin":  # macOS
            total_bytes = float(subprocess.check_output(["sysctl", "-n", "hw.memsize"]).decode().strip())
            return round(total_bytes / (1024**3), 2)
    except Exception:
        pass
    return "Unknown"


def check_versions():
    """Prints the versions of PyTorch, Torchvision, and CUDA."""

    print(f"Python Version: {sys.version}")
    print("-" * 30)

    # Check PyTorch
    print(f"PyTorch Version: {torch.__version__}")
    print("-" * 30)

    # Check Torchvision
    print(f"Torchvision Version: {torchvision.__version__}")
    print("-" * 30)

    print(f"Reading hardware info (OS: {platform.system()})...")

    cpu_name = get_cpu_name()
    ram_gb = get_ram_gb()

    print("-" * 40)
    print(f"CPU Model:  {cpu_name}")
    print(f"Total RAM:  {ram_gb} GB")
    print("-" * 40)

    # --- Check CUDA ---
    print("--- CUDA Information ---")
    if torch.cuda.is_available():
        print("CUDA is AVAILABLE")
        # This is the CUDA version PyTorch was compiled with
        print(f"PyTorch-linked CUDA Version: {torch.version.cuda}")

        # Get details for each available GPU
        device_count = torch.cuda.device_count()
        print(f"Detected {device_count} CUDA-capable device(s).")
        for i in range(device_count):
            print(f"  Device {i}: {torch.cuda.get_device_name(i)}")
            print(f"    Total memory: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.2f} GB")
    else:
        print("CUDA is NOT available.")
        print("PyTorch is running in CPU-only mode.")
    print("-" * 30)


## 1. Laptop Compatibility

In [3]:
check_versions()

Python Version: 3.10.20 (main, Jun 11 2026, 15:17:37) [GCC 14.3.0]
------------------------------
PyTorch Version: 2.9.1+cu128
------------------------------
Torchvision Version: 0.24.1+cu128
------------------------------
Reading hardware info (OS: Linux)...
----------------------------------------
CPU Model:  AMD Ryzen 9 7940HS w/ Radeon 780M Graphics
Total RAM:  30.57 GB
----------------------------------------
--- CUDA Information ---
CUDA is AVAILABLE
PyTorch-linked CUDA Version: 12.8
Detected 1 CUDA-capable device(s).
  Device 0: NVIDIA GeForce RTX 4060 Laptop GPU
    Total memory: 8.00 GB
------------------------------


# 2. Destop Compatibility

In [3]:
check_versions()

Python Version: 3.10.19 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 16:41:31) [MSC v.1929 64 bit (AMD64)]
------------------------------
PyTorch Version: 2.9.1+cu128
------------------------------
Torchvision Version: 0.24.1+cu128
------------------------------
Reading hardware info...
----------------------------------------
CPU Model:  Intel(R) Core(TM) Ultra 7 265K
Total RAM:  63.71 GB
----------------------------------------
--- CUDA Information ---
CUDA is AVAILABLE
PyTorch-linked CUDA Version: 12.8
Detected 1 CUDA-capable device(s).
  Device 0: NVIDIA GeForce RTX 5070 Ti
    Total memory: 15.92 GB
------------------------------
